# Surface Code QEC with Stim & Sinter in Maestro 0.3.1

This notebook demonstrates Maestro 0.3.1's native **Stim and Sinter integration** () for quantum error correction (QEC) benchmarking with PyMatching.

* **Act 1 — The Drop-In QEC Engine**: Generate rotated surface code circuits with Stim, benchmark decoding performance with Sinter, and use Maestro's Matrix Product State () as a custom sampler backend.
* **Act 2 — Beyond-Clifford Noise**: Real hardware experiences coherent over-rotations and idle dephasing. Stim cannot model non-Clifford rotations, but Maestro's tensor-network engine does, revealing how physical noise shifts the error threshold.

In [ ]:
import os, sys
if os.path.isdir("surface_code_noise") and "surface_code_noise" not in sys.path:
    sys.path.insert(0, os.path.abspath("surface_code_noise"))
import time
import numpy as np
import matplotlib.pyplot as plt
import stim
import sinter
import pymatching
import maestro
from maestro.sinter import MaestroSinterSampler, translate_stim_to_maestro
from qec_plotting import plot_threshold_curves, plot_noise_comparison, plot_threshold_shift

print(f"Stim version:       {stim.__version__}")
print(f"Sinter version:     {sinter.__version__}")
print(f"PyMatching version: {pymatching.__version__}")
print(f"Maestro GPU Avail:  {maestro.is_gpu_available()}")


---
## Act 1: Drop-In Sinter QEC Benchmarking with Maestro MPS

We define rotated surface code memory tasks across code distances =3$ and =5$ over a sweep of physical error rates using Stim's circuit generator, and collect decoding statistics via Sinter.

In [ ]:
distances = [3, 5]
physical_error_rates = [0.003, 0.004, 0.005, 0.006, 0.007]
shots = 50
chi = 32

tasks = []
for d in distances:
    for p in physical_error_rates:
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            distance=d,
            rounds=d,
            after_clifford_depolarization=p,
            after_reset_flip_probability=2 * p,
            before_measure_flip_probability=5 * p,
            before_round_data_depolarization=0.1 * p,
        )
        tasks.append(sinter.Task(circuit=circuit, decoder="maestro", json_metadata={"d": d, "p": p, "sampler": "maestro"}))

custom_decoders = {"maestro": MaestroSinterSampler(chi=chi, decoder="pymatching")}

print("Collecting Sinter samples with Maestro MPS sampler (SI1000)...")
t0 = time.perf_counter()
stats = sinter.collect(
    num_workers=min(4, os.cpu_count() or 1),
    max_shots=shots,
    tasks=tasks,
    custom_decoders=custom_decoders,
    start_batch_size=shots,
    max_batch_seconds=120,
    print_progress=False,
)
print(f"Completed in {time.perf_counter() - t0:.2f}s")
print()
print(f"{'Distance':<10} {'Physical Err':<14} {'Shots':<10} {'Errors':<10} {'Logical P_L':<14}")
print("-" * 58)
for s in sorted(stats, key=lambda x: (x.json_metadata["d"], x.json_metadata["p"])):
    d = s.json_metadata["d"]
    p = s.json_metadata["p"]
    p_l = s.errors / max(1, s.shots)
    print(f"d = {d:<6} {p:<14.4f} {s.shots:<10} {s.errors:<10} {p_l:<14.6f}")


### Act 1 Threshold Visualization
Plot the logical error rate vs physical error rate curves for distances =3$ and =5$.

In [ ]:
plot_threshold_curves(stats, "qec_threshold_curve.png")

img = plt.imread("qec_threshold_curve.png")
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.show()

---
## Act 2: Beyond-Clifford Noise (Coherent & Idle Simulation)

Stim's stabilizer tableau simulator cannot simulate continuous rotations. With Maestro 0.3.1, we attach a  with **coherent rotation noise** and **idle decoherence** () to . Sinter and PyMatching decode the resulting syndromes, revealing the physical threshold shift.

In [ ]:
distances = [3, 5]
physical_error_rates = [0.003, 0.004, 0.005, 0.006, 0.007]
shots = 50
chi = 32

coherent_tasks = []
custom_decoders = {}
for d in distances:
    nq = 2 * (d ** 2) - 1
    for p in physical_error_rates:
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            distance=d,
            rounds=d,
            after_clifford_depolarization=p,
            after_reset_flip_probability=2 * p,
            before_measure_flip_probability=5 * p,
            before_round_data_depolarization=0.1 * p,
        )
        _, nm, _ = translate_stim_to_maestro(circuit)
        eps = 1.15 * 2.0 * np.arcsin(np.sqrt(p))
        for q in range(nq):
            nm.set_coherent_rotation(q, eps, 0.0, eps)
            nm.set_idle_noise(q, t1=50e-6, t2=25e-6)
        dec_name = f"maestro_coh_d{d}_p{int(p*10000)}"
        coherent_tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=dec_name,
                json_metadata={"d": d, "p": p, "sampler": "coherent", "num_qubits": nq},
            )
        )
        custom_decoders[dec_name] = MaestroSinterSampler(chi=chi, noise_model=nm, decoder="pymatching")

print("Sampling Coherent + Idle noise with Maestro NoiseModel across code distances...")
t0 = time.perf_counter()
coh_d3 = [t for t in coherent_tasks if t.json_metadata["d"] == 3]
coh_d5 = [t for t in coherent_tasks if t.json_metadata["d"] == 5]
coherent_stats = []
if coh_d3:
    s3 = sinter.collect(
        num_workers=min(len(coh_d3), 4),
        max_shots=100,
        tasks=coh_d3,
        custom_decoders=custom_decoders,
        start_batch_size=100,
        max_batch_seconds=120,
        print_progress=False,
    )
    coherent_stats.extend(s3)
if coh_d5:
    s5 = sinter.collect(
        num_workers=min(len(coh_d5), 4),
        max_shots=40,
        tasks=coh_d5,
        custom_decoders=custom_decoders,
        start_batch_size=40,
        max_batch_seconds=180,
        print_progress=False,
    )
    coherent_stats.extend(s5)

print(f"Completed in {time.perf_counter() - t0:.2f}s")
print()
print(f"{'Distance':<10} {'Phys Err (p)':<14} {'Coherent P_L':<16}")
print("-" * 42)
for s in sorted(coherent_stats, key=lambda x: (x.json_metadata["d"], x.json_metadata["p"])):
    d = s.json_metadata["d"]
    p = s.json_metadata["p"]
    p_l = s.errors / max(1, s.shots)
    print(f"d = {d:<6} {p:<14.4f} {p_l:<16.6f}")


### Step 4: Visualizing the Stacked Threshold Shift & The Pauli Illusion Zone
Stack the coherent noise curves directly onto the Pauli error baseline curves to reveal
where and why PyMatching fails under coherent noise. In the shaded **Pauli Illusion Zone**,
standard Pauli simulations predict that scaling from d=3 to d=5 suppresses errors, whereas
real coherent noise causes d=5 to fail with a higher logical error rate.


In [ ]:
fig_path = plot_threshold_shift(stats, coherent_stats, "qec_threshold_shift.png")
from IPython.display import Image, display
display(Image(filename="qec_threshold_shift.png"))


---
## Act 3: Driving the Nail Further — Deterministic Density Matrix Simulation with Maestro MPO

Why do standard QEC decoders like PyMatching fail when coherent noise is present?

1. **Decoder Assumption**: PyMatching and standard QEC decoders formulate error correction as Minimum-Weight Perfect Matching on a syndrome graph where edge weights assume independent, identically distributed Pauli errors.
2. **Coherent Reality**: Real physical qubits suffer from systematic unitary over-rotations. Unitary rotations do not collapse into independent Pauli flips. Instead, continuous phase errors interfere **constructively** round after round:
   - Incoherent Pauli noise decays **diffusively** (random walk, linear in rounds: infidelity proportional to r * p).
   - Coherent noise accumulates **constructively** (coherent rotation angle theta_total = r * theta, leading to quadratic infidelity growth proportional to r^2 * theta^2).

To prove this physics without Monte Carlo sampling variance, we leverage Maestro's **Matrix Product Operator (MPO)** simulator (`SimulationType.MatrixProductOperator`). MPO simulates the exact mixed-state density matrix rho deterministically, evaluating the exact expectation value <Z_L> of the surface code logical operator across syndrome extraction cycles with **zero shot noise**.

**The Density Matrix Memory Wall**:
Simulating a 25-qubit density matrix (distance-5 data lattice) with standard dense simulators requires 2^50 * 16 bytes = **16.8 Petabytes of RAM** (physically impossible on any supercomputer). Maestro's MPO compresses the mixed state into a 1D tensor train with bounded bond dimension (chi=32), simulating distance-5 in seconds with only **~1.6 MB of RAM** on a laptop.


In [ ]:
from qec_demo import run_act3_mpo_density_matrix_benchmark
from qec_plotting import plot_mpo_logical_decay
from IPython.display import Image, display

# Run deterministic MPO density matrix benchmark across d=3 (9 qubits) and d=5 (25 qubits)
rounds, mpo_results = run_act3_mpo_density_matrix_benchmark(
    distances=[3, 5],
    rounds=[1, 2, 3, 4, 5],
    chi=64,
)

# Plot dual-panel visualization: Exact Logical State Decay + Density Matrix Memory Wall
mpo_plot_path = plot_mpo_logical_decay(rounds, mpo_results, save_path="qec_mpo_logical_decay.png")
display(Image(filename="qec_mpo_logical_decay.png"))
